# Stochastic Calculus from Zero · Episode 2: Worlds, events and random variables

Companion notebook for the video. Every number that appears on screen or in the narration is produced by running this notebook and read from `build/outputs.json`.

> **The coin world.** A share at 100 that each toss multiplies by 1.2 (heads) or 0.8 (tails), three tosses, eight possible worlds. Heads has a 60% real-world chance; a deposit grows 2% a step. Made up for teaching.
>
> Educational material only, not financial advice.

1. Setup · 2. Last episode's answer · 3. Eight possible worlds · 4. Events are sets of worlds · 5. Expectation · 6. Another random variable: a call · 7. The rules · 8. This episode's question · 9. Exercises · 10. Export

**Running in Google Colab?** Nothing to install: this notebook only needs numpy, pandas and matplotlib, which Colab already has. Run the cells in order.

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import itertools
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

out = {"meta": {
    "series": "Stochastic Calculus from Zero",
    "episode": 2,
    "quantlib_version": "not used",
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}

The building blocks used throughout the series (`episodes/_shared/stochastic.py`):

In [ ]:
# Building blocks for Stochastic Calculus from Zero, written out by hand so every step can be read. Copied verbatim
# into each notebook by make_notebook.py, so every notebook runs on its own. Needs: itertools, numpy as np.
#   Coin world (Part 1): a share at 100 that each toss multiplies by 1.2 (heads) or 0.8 (tails), three tosses.
#   Real-world chance of heads 0.6; a deposit grows 2% a step, so the risk-neutral chance is 0.55. Made up for
#   teaching.

class CoinWorld:
    """Every possible world of a few coin tosses, with real-world (P) and risk-neutral (Q) weights.

    A world is a string such as "HTH". Its first n letters are what you know after n tosses, so the block of
    information at time n that a world sits in is simply world[:n].
    """

    def __init__(self, s0=100.0, up=1.2, down=0.8, rate=0.02, p=0.6, steps=3):
        self.s0, self.up, self.down, self.rate, self.p, self.steps = s0, up, down, rate, p, steps
        self.q = (1 + rate - down) / (up - down)   # the odds that make the discounted share a martingale
        self.worlds = ["".join(w) for w in itertools.product("HT", repeat=steps)]

    def weight(self, world, measure="P"):
        heads = self.p if measure == "P" else self.q
        return float(np.prod([heads if c == "H" else 1 - heads for c in world]))

    def path(self, world):
        """Share prices S0, S1, ..., one per toss, along this world."""
        s = [self.s0]
        for c in world:
            s.append(s[-1] * (self.up if c == "H" else self.down))
        return s

    def price(self, world, n):
        return self.path(world)[n]

    def expect(self, f, measure="P"):
        return sum(self.weight(w, measure) * f(w) for w in self.worlds)

    def given(self, n, f, measure="P", by=None):
        """E[f | information at time n], as {block: best guess}. `by` groups worlds some other way (e.g. by price)."""
        key = by or (lambda w: w[:n])
        blocks = {}
        for w in self.worlds:
            blocks.setdefault(key(w), []).append(w)
        return {b: sum(self.weight(w, measure) * f(w) for w in ws) / sum(self.weight(w, measure) for w in ws)
                for b, ws in blocks.items()}

    def table(self, **extra):
        """One row per world (HHH ... TTT): weights, prices S0..Sn, number of heads, plus extra columns
        given as functions of the world, e.g. table(call=lambda w: max(cw.price(w, 3) - 100, 0))."""
        rows = []
        for w in self.worlds:
            row = {"world": w, "P": self.weight(w, "P"), "Q": self.weight(w, "Q"), "heads": w.count("H")}
            row |= {f"S{n}": s for n, s in enumerate(self.path(w))}
            row |= {k: f(w) for k, f in extra.items()}
            rows.append(row)
        return rows

    def blocks(self, n):
        """The blocks of information at time n: lists of worlds you cannot yet tell apart."""
        out = {}
        for w in self.worlds:
            out.setdefault(w[:n], []).append(w)
        return list(out.values())


def brownian_paths(n_paths, steps, years, rng):
    """n_paths Brownian paths on a grid of `steps` equal steps: returns (t, W, dW), W starting at 0."""
    dt = years / steps
    dW = rng.standard_normal((n_paths, steps)) * np.sqrt(dt)
    W = np.concatenate([np.zeros((n_paths, 1)), dW.cumsum(axis=1)], axis=1)
    return np.linspace(0, years, steps + 1), W, dW


def sum_of_squares(increments):
    """Sum of squared steps along each path (the quadratic variation on this grid)."""
    return (np.asarray(increments) ** 2).sum(axis=-1)


def chain_rule_sum(W, dW):
    """The school chain rule's prediction for W_T^2: add up 2 W dW, with W taken at the start of each step."""
    return 2 * (W[..., :-1] * dW).sum(axis=-1)

## 2. Last episode's answer

Run a Brownian path for two years and the sum of squared steps settles at the time elapsed: 2.

In [ ]:
out["prev"] = {"years": 2, "answer": 2}

## 3. Eight possible worlds

Three tosses of a coin give $2^3 = 8$ possible worlds, from HHH to TTT. That list is the **sample space**. Each world gets a weight: heads has a 60% chance, so HHH has weight $0.6^3 = 0.216$ and TTT has $0.4^3 = 0.064$. The share starts at 100 and each toss multiplies it by 1.2 (heads) or 0.8 (tails). A **random variable** is just a number attached to each world: here the prices $S_1, S_2, S_3$, the number of heads, and (Section 6) a call's payoff.

In [ ]:
cw = CoinWorld()
K = 100.0
rows = cw.table(call=lambda w: max(cw.price(w, 3) - K, 0.0))
assert abs(sum(r["P"] for r in rows) - 1) < 1e-12
out["coin"] = {"worlds": rows, "n_worlds": len(rows), "p": cw.p, "up": cw.up, "down": cw.down, "s0": cw.s0, "strike": K,
               "w_HHH": rows[0]["P"], "w_HHT": rows[1]["P"], "w_TTT": rows[-1]["P"],
               "S3_values": sorted({round(r["S3"], 6) for r in rows}, reverse=True)}
pd.DataFrame(rows)

## 4. Events are sets of worlds

An **event** is a set of worlds, and its probability is the total weight of those worlds. "The share ends above 100" happens in the four worlds with at least two heads.

In [ ]:
above = [r["world"] for r in rows if r["S3"] > 100]
two_heads = [r["world"] for r in rows if r["heads"] == 2]
out["events"] = {"above_100": above, "p_above_100": sum(r["P"] for r in rows if r["S3"] > 100), "n_above": len(above),
                 "two_heads": two_heads, "p_two_heads": sum(r["P"] for r in rows if r["heads"] == 2),
                 "p_below_100": sum(r["P"] for r in rows if r["S3"] < 100)}
assert abs(out["events"]["p_above_100"] + out["events"]["p_below_100"] - 1) < 1e-12
assert set(above) == {"HHH", "HHT", "HTH", "THH"}
out["events"]

## 5. Expectation

The expectation is the weighted average over all worlds: $E_P[S_3] = \sum_\omega P(\omega)\,S_3(\omega)$. Each toss multiplies the share by 1.2 or 0.8, on average by $0.6 \times 1.2 + 0.4 \times 0.8 = 1.04$, so $E_P[S_3] = 100 \times 1.04^3$.

In [ ]:
e_s3 = cw.expect(lambda w: cw.price(w, 3))
sd_s3 = math.sqrt(cw.expect(lambda w: (cw.price(w, 3) - e_s3) ** 2))
growth = cw.p * cw.up + (1 - cw.p) * cw.down
assert abs(e_s3 - 100 * growth ** 3) < 1e-9
out["expect"] = {"E_S3": e_s3, "sd_S3": sd_s3, "growth": growth, "E_S1": cw.expect(lambda w: cw.price(w, 1))}
out["expect"]

## 6. Another random variable: a call

A call struck at 100 on the final price pays $\max(S_3 - 100, 0)$: one more column, one more number per world. Its expectation under the real-world weights is computed the same way (Episode 6 shows why that is *not* its price).

In [ ]:
out["call"] = {"payoffs": sorted({round(r["call"], 6) for r in rows}, reverse=True), "E_call": cw.expect(lambda w: max(cw.price(w, 3) - K, 0))}
out["call"]

## 7. The rules

Andrey Kolmogorov's 1933 book *Grundbegriffe der Wahrscheinlichkeitsrechnung* set out the rules every probability model obeys: weights are never negative, all of them together add up to 1, and the probability of an event made of separate pieces is the sum of the pieces. Our coin world obeys all three.

In [ ]:
w = {r["world"]: r["P"] for r in rows}
assert min(w.values()) >= 0 and abs(sum(w.values()) - 1) < 1e-12
assert abs(out["events"]["p_above_100"] - (w["HHH"] + out["events"]["p_two_heads"])) < 1e-12   # 3 heads + exactly 2 heads
out["history"] = {"kolmogorov": 1933}

## 8. This episode's question

What's the chance of exactly two heads? Three worlds (HHT, HTH, THH), each with weight $0.6^2 \times 0.4 = 0.144$: in total 0.432.

In [ ]:
out["question"] = {"answer": out["events"]["p_two_heads"], "each": rows[1]["P"], "n": len(two_heads)}
assert abs(out["question"]["answer"] - 0.432) < 1e-12

## 9. Exercises

Three to do by hand, two in code. Try each before opening its solution. The code solutions run with the rest of the notebook, so their checks are tested every time.

**Exercise 1 (By hand).** What is the chance of at least one tail in three tosses?

<details><summary>Answer</summary>

Everything except HHH: $1 - 0.6^3 = 1 - 0.216 = 0.784$.

</details>

**Exercise 2 (By hand).** What is the expected number of heads, $E_P[\text{heads}]$?

<details><summary>Answer</summary>

Each toss adds 0.6 heads on average, so $3 \times 0.6 = 1.8$. (Or add up weight × heads over the 8 worlds.)

</details>

**Exercise 3 (By hand).** Is "$S_2 = 96$" an event? Which worlds make it up, and what is its probability?

<details><summary>Answer</summary>

Yes: every set of worlds is an event here. $S_2 = 96$ needs one head and one tail in the first two tosses: HTH, HTT, THH, THT, with total weight $2 \times 0.6 \times 0.4 = 0.48$.

</details>

**Exercise 4 (In code).** Build the same table for **four** tosses. Check that the weights add up to 1 and that $E_P[S_4] = 100 \times 1.04^4$.

In [ ]:
#@title Solution 4 (double-click to show the code) { display-mode: "form" }
cw4 = CoinWorld(steps=4)
t4 = cw4.table()
assert len(t4) == 16 and abs(sum(r["P"] for r in t4) - 1) < 1e-12
assert abs(cw4.expect(lambda w: cw4.price(w, 4)) - 100 * 1.04 ** 4) < 1e-9
print(len(t4), "worlds; E_P[S4] =", round(cw4.expect(lambda w: cw4.price(w, 4)), 4))

**Exercise 5 (In code).** Compute $E_P[\text{call}]$ and the probability that the call pays anything.

In [ ]:
#@title Solution 5 (double-click to show the code) { display-mode: "form" }
e_call = cw.expect(lambda w: max(cw.price(w, 3) - K, 0))
p_pays = cw.expect(lambda w: float(cw.price(w, 3) > K))
assert abs(p_pays - 0.648) < 1e-12
print(f"E_P[call] = {e_call:.4f}; chance it pays = {p_pays:.3f}")

## 10. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")